# PRISM - Camelyon17 figures

Three figures, styled for print rather than for a screen.

| figure | what it carries |
|---|---|
| `fig_contrast.pdf` | target ECE against label fraction, the two designs side by side |
| `fig_mechanism.pdf` | ECE slope against transfer AUROC, all 192 combinations |
| `fig_hospital_matrix.pdf` | AUROC and ECE by source and target hospital |

## Design choices

**Direct labelling instead of legends.** A legend under eight lines makes the
eye travel back and forth. Labels sit at the end of each line, spread apart by
a deterministic routine rather than by `adjustText`, so the output is
reproducible run to run.

**Highlight and mute.** Eight equally saturated lines are noise. The models the
claim concerns are coloured; the rest are grey and unlabelled but still drawn,
so the reader can see the spread without being asked to track it.

**Left-aligned titles with a subtitle line**, spines only on the left and
bottom, horizontal grid only, and `constrained_layout` for spacing. These are
the differences between a figure that looks like a default plot and one that
looks like it belongs in a journal.

Colourblind-safe throughout (Okabe--Ito), verified under deuteranopia,
protanopia and greyscale in the last cell. CPU, under a minute.

In [1]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.stats import spearmanr
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/PRISM'
OUT_DIR = f'{BASE}/results_v2'
FIG_DIR = f'{BASE}/figures_v3'
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 400, 'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'font.size': 8.5, 'axes.labelsize': 8.5, 'axes.titlesize': 9.5,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 7.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.7, 'axes.labelpad': 5,
    'axes.titlelocation': 'left', 'axes.titlepad': 16,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'xtick.major.size': 3, 'ytick.major.size': 3,
    'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
    'axes.grid': True, 'axes.grid.axis': 'y',
    'grid.color': '#DDDDDD', 'grid.linewidth': 0.6,
    'legend.frameon': False,
    'figure.constrained_layout.use': True,
})

# Okabe-Ito
C = {'orange': '#E69F00', 'skyblue': '#56B4E9', 'green': '#009E73',
     'yellow': '#F0E442', 'blue': '#0072B2', 'vermillion': '#D55E00',
     'purple': '#CC79A7', 'grey': '#8C8C8C'}
MUTED = '#C7C7C7'
HILITE = {'UNI': C['green'], 'VIRCHOW2': C['blue'], 'GigaPath': C['skyblue'],
          'H-Optimus-0': C['purple'], 'CLIP': C['vermillion']}
MODELS = ['CLIP','PLIP','CONCH','VIRCHOW2','UNI','GigaPath','H-Optimus-0','MIDNIGHT']

lab  = pd.read_csv(f'{OUT_DIR}/ood_all_v2.csv')
cov  = pd.read_csv(f'{OUT_DIR}/camelyon17_transfer.csv')
mech = pd.read_csv(f'{OUT_DIR}/mechanism_summary.csv')


def spread(vals, gap):
    """Push overlapping label positions apart while preserving order.

    Deterministic, unlike adjustText, so the figure is identical run to run."""
    order = sorted(vals, key=vals.get)
    out, prev = {}, -1e18
    for k in order:
        y = max(vals[k], prev + gap)
        out[k] = y
        prev = y
    shift = np.mean(list(vals.values())) - np.mean(list(out.values()))
    return {k: v + shift for k, v in out.items()}


print(f'label design {len(lab):,} rows, covariate {len(cov):,} rows, '
      f'mechanism {len(mech)} combinations')

Mounted at /content/drive
label design 576 rows, covariate 2,880 rows, mechanism 192 combinations


## 1. The contrast

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0), sharey=True,
                         width_ratios=[1, 1.22])

for ax, df, title, sub in [
        (axes[0], lab, 'Label-definition shift',
         '4 pairs  ·  transfer AUROC $\\approx$ 0.5'),
        (axes[1], cov, 'Covariate shift only',
         '20 hospital pairs  ·  AUROC 0.85–0.99')]:
    for m in MODELS:
        if m in HILITE:
            continue
        s = df[df.model == m].groupby('fraction')['ece_fixed'].mean().sort_index()
        if not s.empty:
            ax.plot(s.index * 100, s.values, lw=0.9, color=MUTED, zorder=1)
    for m, col in HILITE.items():
        s = df[df.model == m].groupby('fraction')['ece_fixed'].mean().sort_index()
        if s.empty:
            continue
        ax.plot(s.index * 100, s.values, lw=1.7, color=col, zorder=3,
                solid_capstyle='round')
        ax.plot([100], [s.iloc[-1]], 'o', ms=3.4, color=col, zorder=4)
    ax.set_xscale('log')
    ax.set_xticks([1, 5, 10, 25, 50, 100])
    ax.set_xticklabels(['1', '5', '10', '25', '50', '100'])
    ax.set_xlim(0.9, 108)
    ax.set_xlabel('source label fraction (%)')
    ax.set_title(title, loc='left', fontweight='semibold')
    ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=7.4,
            color='#7A7A7A', va='bottom')

axes[0].set_ylabel('target ECE (raw)')

# direct labels on the right panel
last = {m: cov[(cov.model == m) & (cov.fraction == 1.0)]['ece_fixed'].mean()
        for m in MODELS}
pos = {m: last[m] for m in HILITE}
pos['others'] = float(np.mean([last[m] for m in MODELS if m not in HILITE]))
ylo, yhi = axes[1].get_ylim()
pos = spread(pos, gap=0.062 * (yhi - ylo))
for k, y in pos.items():
    col = HILITE.get(k, '#9A9A9A')
    y0 = last.get(k, y)
    axes[1].plot([100, 112], [y0, y], lw=0.6, color=col, alpha=0.55,
                 clip_on=False, zorder=2)
    axes[1].text(115, y, k, fontsize=7.3, color=col, va='center', ha='left',
                 fontweight='medium' if k in HILITE else 'normal')

for ax, txt, col, y in [
        (axes[0], 'more labels →\nworse calibration', C['vermillion'], 0.82),
        (axes[1], 'more labels →\nbetter calibration', C['blue'], 0.15)]:
    ax.text(0.04, y, txt, transform=ax.transAxes, fontsize=7.6, color=col,
            va='center', linespacing=1.4)

fig.savefig(f'{FIG_DIR}/fig_contrast.pdf')
fig.savefig(f'{FIG_DIR}/fig_contrast.png')
plt.show()

for name, df in [('label-definition', lab), ('covariate', cov)]:
    s = df.groupby('fraction')['ece_fixed'].mean()
    print(f'  {name:<18} ECE {s.iloc[0]:.3f} -> {s.iloc[-1]:.3f}  '
          f'({s.iloc[-1] - s.iloc[0]:+.3f})')

  label-definition   ECE 0.201 -> 0.283  (+0.082)
  covariate          ECE 0.248 -> 0.099  (-0.149)


## 2. The mechanism

A density strip above the panel shows where each design sits on the AUROC axis,
so the reader can see that they overlap in the transition zone rather than
separating cleanly. That overlap is what answers the dataset confound.

In [3]:
fig = plt.figure(figsize=(5.6, 4.3))
gs = GridSpec(2, 1, height_ratios=[1, 5.2], hspace=0.06, figure=fig)
axr = fig.add_subplot(gs[0])
ax  = fig.add_subplot(gs[1], sharex=axr)

ST = {'label-definition': dict(marker='s', color=C['vermillion'],
                               label='label-definition shift  (4 pairs)'),
      'covariate':        dict(marker='o', color=C['blue'],
                               label='covariate shift  (20 hospital pairs)')}

for d, st in ST.items():
    v = mech[mech.design == d]['auroc_mean']
    axr.scatter(v, np.full(len(v), 0.5 if d == 'covariate' else 1.4), s=13,
                alpha=0.5, marker=st['marker'], color=st['color'],
                edgecolors='none', clip_on=False)
axr.set_ylim(0, 2)
axr.axis('off')

for d, st in ST.items():
    dd = mech[mech.design == d]
    ax.scatter(dd['auroc_mean'], dd['ece_slope'], s=27, alpha=0.66,
               edgecolors='white', linewidths=0.45, zorder=3, **st)

ax.axhline(0, color='#333333', lw=0.9, zorder=1)
ax.axvline(0.85, color=C['green'], lw=1.1, ls=(0, (4, 2.5)), zorder=1)
lo, hi = ax.get_ylim()
ax.axhspan(0, hi, color=C['vermillion'], alpha=0.05, zorder=0)
ax.set_ylim(lo, hi)
ax.text(0.851, hi * 0.97, ' effect threshold\n AUROC $\\approx$ 0.85',
        fontsize=7.4, color=C['green'], va='top', ha='left', linespacing=1.5)

r_all = spearmanr(mech['auroc_mean'], mech['ece_slope']).correlation
cvm = mech[mech.design == 'covariate']
r_cov = spearmanr(cvm['auroc_mean'], cvm['ece_slope']).correlation
ax.text(0.012, 0.10,
        f'Spearman $\\rho$\nall {len(mech)} combinations   {r_all:.2f}\n'
        f'within Camelyon17 only   {r_cov:.2f}',
        transform=ax.transAxes, fontsize=7.4, va='bottom', ha='left',
        color='#444444', linespacing=1.6)

ax.text(0.012, 0.965, 'more labels worsen calibration',
        transform=ax.transAxes, fontsize=7.6, color=C['vermillion'], va='top')
ax.text(0.988, 0.035, 'more labels improve calibration',
        transform=ax.transAxes, fontsize=7.6, color=C['blue'],
        va='bottom', ha='right')

ax.set_xlabel('transfer AUROC at the target')
ax.set_ylabel('slope of target ECE  in log$_{10}$ label fraction')
axr.set_title('The sign of the effect follows transfer quality',
              loc='left', fontweight='semibold')
ax.legend(loc='upper right', bbox_to_anchor=(1.0, 0.94), handletextpad=0.35)
plt.setp(axr.get_xticklabels(), visible=False)

fig.savefig(f'{FIG_DIR}/fig_mechanism.pdf')
fig.savefig(f'{FIG_DIR}/fig_mechanism.png')
plt.show()

print(f'  Spearman all {len(mech)}: {r_all:+.3f}   Camelyon17 only: {r_cov:+.3f}')
band = mech[(mech.auroc_mean >= 0.55) & (mech.auroc_mean < 0.85)]
print(f'  transition zone 0.55-0.85: {len(band)} combinations from '
      f'{band.design.nunique()} design(s), {100*band.rises.mean():.0f}% rise')

  Spearman all 192: -0.782   Camelyon17 only: -0.647
  transition zone 0.55-0.85: 24 combinations from 2 design(s), 88% rise


## 3. Hospital matrix

Marginal bars carry the point: the column means vary more than the row means,
so difficulty belongs to the target hospital.

In [4]:
full = cov[cov.fraction == 1.0]
HOSP = sorted(full['src'].unique())

fig = plt.figure(figsize=(7.4, 3.4))
outer = GridSpec(1, 2, figure=fig, wspace=0.28)

for k, (metric, cmap, title) in enumerate([
        ('auroc', 'Blues', 'transfer AUROC'),
        ('ece_fixed', 'Oranges', 'target ECE (raw)')]):
    gs = outer[k].subgridspec(2, 2, height_ratios=[1, 5], width_ratios=[5, 1],
                              hspace=0.05, wspace=0.05)
    axm = fig.add_subplot(gs[1, 0])
    axt = fig.add_subplot(gs[0, 0], sharex=axm)
    axs = fig.add_subplot(gs[1, 1], sharey=axm)

    M = (full.pivot_table(index='src', columns='tgt', values=metric)
         .reindex(index=HOSP, columns=HOSP))
    V = M.values
    vmin, vmax = np.nanmin(V), np.nanmax(V)
    axm.imshow(V, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    for i in range(len(HOSP)):
        for j in range(len(HOSP)):
            v = V[i, j]
            if np.isnan(v):
                axm.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                            fc='#F4F4F4', ec='white', lw=1.6))
                continue
            rel = (v - vmin) / (vmax - vmin + 1e-12)
            axm.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=7,
                     color='white' if rel > 0.6 else '#333333')
    axm.set_xticks(range(len(HOSP))); axm.set_xticklabels([f'h{h}' for h in HOSP])
    axm.set_yticks(range(len(HOSP))); axm.set_yticklabels([f'h{h}' for h in HOSP])
    axm.set_xlabel('target hospital'); axm.set_ylabel('source hospital')
    axm.grid(False)
    for sp in axm.spines.values():
        sp.set_visible(False)

    col_m, row_m = np.nanmean(V, 0), np.nanmean(V, 1)
    axt.bar(range(len(HOSP)), col_m - vmin, bottom=vmin, width=0.62,
            color='#6B6B6B', linewidth=0)
    axt.set_ylim(vmin, vmax); axt.axis('off')
    axt.set_title(title, loc='left', fontweight='semibold', pad=22)
    axt.text(0, 1.10, f'spread of means:  target {col_m.std():.4f}  ·  '
             f'source {row_m.std():.4f}', transform=axt.transAxes,
             fontsize=7, color='#7A7A7A', va='bottom')

    axs.barh(range(len(HOSP)), row_m - vmin, left=vmin, height=0.62,
             color='#B4B4B4', linewidth=0)
    axs.set_xlim(vmin, vmax); axs.axis('off')

fig.savefig(f'{FIG_DIR}/fig_hospital_matrix.pdf')
fig.savefig(f'{FIG_DIR}/fig_hospital_matrix.png')
plt.show()

M = full.pivot_table(index='src', columns='tgt', values='auroc')
print(f'  AUROC: spread of source means {M.mean(axis=1).std():.4f}, '
      f'target means {M.mean(axis=0).std():.4f}')

  AUROC: spread of source means 0.0077, target means 0.0084


## 4. Accessibility

In [5]:
from PIL import Image

def simulate(path, kind):
    im = np.asarray(Image.open(path).convert('RGB'), dtype=float) / 255
    if kind == 'greyscale':
        g = im @ [0.2126, 0.7152, 0.0722]
        return np.stack([g] * 3, -1)
    M = {'deuteranopia': [[0.625, 0.375, 0], [0.7, 0.3, 0], [0, 0.3, 0.7]],
         'protanopia':   [[0.567, 0.433, 0], [0.558, 0.442, 0],
                          [0, 0.242, 0.758]]}[kind]
    return np.clip(im @ np.array(M).T, 0, 1)

for name in ['fig_contrast', 'fig_mechanism', 'fig_hospital_matrix']:
    p = f'{FIG_DIR}/{name}.png'
    f2, axs = plt.subplots(1, 4, figsize=(12, 2.6))
    f2.set_constrained_layout(False)
    axs[0].imshow(np.asarray(Image.open(p).convert('RGB')))
    axs[0].set_title(f'{name}  (original)', fontsize=8)
    for a, k in zip(axs[1:], ['deuteranopia', 'protanopia', 'greyscale']):
        a.imshow(simulate(p, k)); a.set_title(k, fontsize=8)
    for a in axs:
        a.axis('off'); a.grid(False)
    plt.show()

print('=== files written ===')
for f in sorted(os.listdir(FIG_DIR)):
    print(f'  {f:<30} {os.path.getsize(f"{FIG_DIR}/{f}")/1024:>7.1f} KB')
print('\nUpload the three .pdf files to the Overleaf figures/ folder.')

=== files written ===
  fig_contrast.pdf                  28.4 KB
  fig_contrast.png                 334.2 KB
  fig_hospital_matrix.pdf           44.9 KB
  fig_hospital_matrix.png          232.5 KB
  fig_mechanism.pdf                 32.4 KB
  fig_mechanism.png                287.9 KB

Upload the three .pdf files to the Overleaf figures/ folder.


## 5. LaTeX to add

Contrast figure before the reverse-scaling paragraph, mechanism after it,
matrix in the Camelyon17 appendix.

```latex
\begin{figure}[t]
\centering
\includegraphics[width=\linewidth]{figures/fig_contrast.pdf}
\caption{\textbf{The same protocol, opposite directions.} Raw target ECE
against source label fraction, averaged over transfer pairs within each design.
Coloured lines are the four models the original claim covers together with
CLIP, the one model that shows the effect under covariate shift; grey lines are
the remaining three. Left: the four pairs of the original design, where the
positive class is redefined across the pair and probes transfer at AUROC near
0.5. Right: twenty directed hospital pairs from Camelyon17, where task and
label definition are identical at every hospital and only the scanner, staining
protocol and patient population differ.}
\label{fig:contrast}
\end{figure}

\begin{figure}[t]
\centering
\includegraphics[width=0.74\linewidth]{figures/fig_mechanism.pdf}
\caption{\textbf{The sign of the effect is set by transfer quality, not by the
kind of shift.} Each point is one (model, pair) combination; the vertical axis
is the slope of target ECE in $\log_{10}$ label fraction, so points in the
shaded region are combinations where additional source labels worsen
calibration. The strip above the panel marks where each design sits on the
AUROC axis: they overlap through the transition zone rather than separating, so
covariate-shift transfers that perform poorly also show the effect. The
relationship holds within Camelyon17 alone, where dataset, task and kind of
shift are all constant.}
\label{fig:mechanism}
\end{figure}

\begin{figure}[t]
\centering
\includegraphics[width=\linewidth]{figures/fig_hospital_matrix.pdf}
\caption{\textbf{Transfer difficulty belongs to the target hospital.} Transfer
AUROC and raw target ECE at full supervision, averaged over the eight
foundation models, for each ordered pair of Camelyon17 hospitals. Marginal bars
give column and row means. The diagonal is empty because source and target are
always distinct. Column means vary more than row means: which hospital a probe
is deployed to matters more than which it was trained on.}
\label{fig:hospital-matrix}
\end{figure}
```